# НДМ — независимая ML-модель трещинообразования и предела прочности по лучу

**Версия 4 — нормативные характеристики для обеих предельных границ; самостоятельный блокнот, не содержит обучения армирования.** Ширина сечения `b=1000 мм`, силы `N` — кН/м, моменты `My` — кН·м/м, площади **фактически установленного** армирования — см²/м.

Для каждого набора из **11 входов** вычисляются два индекса нагрузки при пропорциональном нагружении `N(λ)=λN`, `My(λ)=λMy`:

- `lambda_ult` — первое пересечение луча с предельной деформационной границей **при нормативных** $R_{b,ser},R_{s,ser}$, **без** работающего растянутого бетона;
- `lambda_crc` — первое пересечение луча с границей трещинообразования **при нормативных** $R_{b,ser},R_{bt,ser},R_{s,ser}$, **с** работающим растянутым бетоном.

**Дополнительные выходы напряжений:** $\sigma_{s,crc}$ — напряжение в растянутой арматуре **непосредственно после образования первой трещины**. Для его вычисления берём $N_{crc}=\lambda_{crc}N$, $M_{crc}=\lambda_{crc}M_y$, отключаем растянутый бетон и повторно решаем равновесие НДМ с нормативными характеристиками. Сохраняются напряжения обоих слоёв $\sigma_{sb,crc}$ и $\sigma_{st,crc}$ (МПа, знак «+» — растяжение); $\sigma_{s,crc}=\max(0,\sigma_{sb,crc},\sigma_{st,crc})$. До предела прочности трещина может не образоваться: в таких строках все три напряжения не определяются (`NaN`). Также вычисляются $\sigma_{sb,full}$ и $\sigma_{st,full}$ при **полной заданной нагрузке** $N,M_y$ (то есть $\lambda=1$) с отключённым растянутым бетоном, по нормативным характеристикам. Для этих напряжений решается отдельное равновесие сечения, а не масштабируются напряжения на трещине. Возвращаем $\sigma_{s,full}=\max(0,\sigma_{sb,full},\sigma_{st,full})$.

**Область применимости $\sigma_{s,full}$:** только если к полной нагрузке трещина уже образовалась ($\lambda_{crc}\le1$) и нормативный предел прочности ещё не достигнут ($\lambda_{ult}\ge1$). Если сечение при полной нагрузке не треснуло, или уже превысило предел, напряжения в *трещиноватом* сечении от полной нагрузки оставляются `NaN` (это не означает нулевое напряжение арматуры). Если численный решатель не подтвердил равновесие, результат также `NaN`.

Отдельные напряжения позволяют выбрать фактически растянутый ряд при расчёте ширины раскрытия.

`lambda_first=min(lambda_ult,lambda_crc)`; события: `0=прочность раньше трещин`, `1=трещины раньше прочности`, `2=практически одновременно`. При отсутствии пересечения с границей трещин до предела прочности значение `lambda_crc` в эталоне допускается как `NaN` (без искусственного большого числа).

**Единая база материалов:** в обеих проверках применяются нормативные характеристики одного класса бетона и арматуры. На прочностной границе растянутый бетон выключен; на границе образования трещин работает. Поэтому сравнение $\lambda_{ult}$ и $\lambda_{crc}$ характеризует порядок **двух расчётных границ** по одному лучу, но само по себе ещё не является доказательством экспериментальной последовательности событий: используются разные идеализации работы растянутого бетона и численное нахождение пересечений. Результаты требуют проверки по НДМ. Для вычисления $\sigma_{s,crc}$ используются те же нормативные характеристики, что и для трещинообразования.



Используются нормативные таблицы СП 63.13330.2018: тяжёлый бетон (6.7, 6.8, 6.11), арматура (6.13, 6.14 с изменением № 1). Параметры редактируются в следующей ячейке. Дополнительные коэффициенты условий работы, длительность воздействия и иные классы бетона/стали здесь **не включены**, если явно не заданы.
Диапазоны привязок арматуры настроены в первой ячейке: для каждого слоя независимо $a_{min}\le a_{sb},a_{st}\le0{,}45h$.

Расчётные $R_b,R_s,R_{sc}$ остаются среди 11 входов для совместимости с первой моделью и **только для выбора нормативной строки** из матриц. Внутри обеих предельных проверок они не используются как сопротивления. Нормативная диаграмма стали здесь принята симметричной по растяжению/сжатию: $R_{sc,ser}=R_{s,ser}$; при необходимости отдельное $R_{sc,ser}$ следует добавить в матрицу арматуры.

In [1]:
# ==========================================================
# НАСТРОЙКИ ВТОРОЙ МОДЕЛИ — ВСЁ РЕДАКТИРОВАТЬ ЗДЕСЬ
# ==========================================================
CR_SEED = 20261028
CR_N_SAMPLES = 300_000           # новая независимая выборка
CR_DATA_FILE = 'ndm_crack_ray_dataset_300k_normative_sigma_crc_full.npz'
CR_MODEL_FILE = 'ndm_crack_ray_mlp_300kb_normative_sigma_crc_full.npz'
CR_B = 1000.0                   # мм = 1 м
CR_H_MIN, CR_H_MAX = 150.0, 1000.0
CR_A_MIN, CR_A_MAX = 0.5, 200.0 # см²/м — КАЖДЫЙ слой
CR_RHO_MIN_PCT = 0.01          # % каждого слоя относительно b*h
CR_RHO_MAX_PCT = 10.00
CR_AS_OFFSET_MIN = 15.0        # мм — минимум привязки a_sb и a_st
CR_AS_OFFSET_MAX_H_FRAC = 0.45  # доля толщины h — максимум каждой привязки
CR_ES = 200000.0               # МПа
CR_ES2 = 0.025                 # ограничение относительной деформации стали
CR_EB0, CR_EB2 = 0.002, 0.0035
CR_ET0, CR_ET2 = 0.00010, 0.00015
CR_LAMBDA_MIN = 0.10           # диапазон индексов в обучающей выборке
CR_LAMBDA_MAX = 20.0
CR_SIMULTANEOUS_REL_TOL = 0.005
CR_K_POINTS_ULT = 121          # узлы огибающей; увеличивайте для точности эталонов
CR_K_POINTS_CRC = 121
CR_REFINE_ITERS = 32
CR_POST_NEWTON_ITERS = 35      # Ньютон при выключенном растянутом бетоне
CR_POST_TOL = 1e-8            # нормированная невязка равновесия
CR_BATCH = 2048
CR_AUDIT_COUNT = 1000

# МАТРИЦА БЕТОНА (тяжёлый): [класс, Rb (I гр), Eb, Rb_ser, Rbt_ser], МПа
CR_CONCRETE_MATRIX = (
    ('B15',  8.5, 24000., 11.0, 1.10),
    ('B20', 11.5, 27500., 15.0, 1.35),
    ('B25', 14.5, 30000., 18.5, 1.55),
    ('B30', 17.0, 32500., 22.0, 1.75),
    ('B35', 19.5, 34500., 25.5, 1.95),
    ('B40', 22.0, 36000., 29.0, 2.10),
    ('B45', 25.0, 37000., 32.0, 2.25),
    ('B50', 27.5, 38000., 36.0, 2.45),
    ('B55', 30.0, 39000., 39.5, 2.60),
    ('B60', 33.0, 39500., 43.0, 2.75),
)
# МАТРИЦА АРМАТУРЫ: [класс, Rs (I гр), Rsc (I гр), Rs_ser (нормативное)], МПа
# В двух расчётах второй ML-модели принимаем Rsc_ser=Rs_ser (симметричная диаграмма).
# А500, Rsc=400 МПа соответствует кратковременному действию;
# при другом режиме таблицу и постановку нужно адаптировать.
CR_STEEL_MATRIX = (
    ('A500', 435.0, 400.0, 500.0),
    ('A400', 340.0, 340.0, 400.0),
)

# Сеть: 22 признака -> 188 -> 188 -> 188 -> 9 (2 λ, 4 σ, 3 события)
# σ: два слоя сразу после трещины и два слоя при полной нагрузке λ=1.
# При λ=1 напряжения определены только для трещиноватого состояния ДО разрушения.
CR_HIDDEN = 188
CR_EPOCHS = 200
CR_PATIENCE = 25
CR_BATCH_SIZE = 1024
CR_LR = 0.001
CR_WEIGHT_SIGMA = 0.20         # вес регрессии напряжений на трещине (маска по валидным меткам)
CR_WEIGHT_EVENTS = 0.30        # вес loss классификации относительно регрессии
CR_TRAIN_PART = 0.80
CR_VAL_PART = 0.10
# Чтобы заново обучить на другой выборке: меняйте seed или размер, файлы перезапишутся.
# Старые файлы никакая ячейка не удаляет.

## 1. Импорты, геометрия и стабильная НДМ

In [2]:
import numpy as np
from pathlib import Path
import time, math, json
from numba import njit, prange
import torch
from torch import nn
print('NumPy:',np.__version__,'Numba:',__import__('numba').__version__,'PyTorch:',torch.__version__)

NumPy: 2.3.5 Numba: 0.63.1 PyTorch: 2.14.1+cu130


In [3]:
"""Extracted stable NDM forward model, no experimental optimizers. Units: N, N*mm, mm, MPa, mm2."""
import numpy as np
from dataclasses import dataclass

@dataclass(frozen=True)
class Concrete:
    Rc: float
    Rt: float
    E: float = 32500.0
    eb0: float = 0.002
    eb2: float = 0.0035
    et0: float = 0.0001
    et2: float = 0.00015

@dataclass(frozen=True)
class Steel:
    Rst: float
    Rsc: float
    E: float = 200000.0
    eult: float = 0.025

@dataclass(frozen=True)
class Section:
    b: float = 140.0
    h: float = 200.0
    a_bot: float = 25.0
    a_top: float = 25.0
    @property
    def zb(self): return -self.h/2 + self.a_bot
    @property
    def zt(self): return self.h/2 - self.a_top
    @property
    def d(self): return self.zt-self.zb

CONC_I = Concrete(17.0, 1.15)
CONC_II = Concrete(22.0, 1.75)
STEEL_I = Steel(435.0,400.0)  # Только значение по умолчанию; генератор использует выбранную строку ML_STEEL_MATRIX.
STEEL_II = Steel(500.0,500.0)
SEC = Section()

def concrete_segments(c: Concrete, include_tension: bool):
    """(branch, strain_low, strain_high, alpha, beta) for sigma=alpha*strain+beta."""
    eb1 = 0.6*c.Rc/c.E
    cb = 0.4*c.Rc/(c.eb0-eb1)
    parts = [
        ('Плато сжатия', -c.eb2, -c.eb0, 0.0, -c.Rc),
        ('Переходное сжатие', -c.eb0, -eb1, cb, -0.6*c.Rc+cb*eb1),
        ('Упругое сжатие', -eb1, 0.0, c.E, 0.0),
    ]
    if include_tension:
        et1 = 0.6*c.Rt/c.E
        ct = 0.4*c.Rt/(c.et0-et1)
        parts += [
            ('Упругое растяжение', 0.0, et1, c.E, 0.0),
            ('Переходное растяжение', et1, c.et0, ct, 0.6*c.Rt-ct*et1),
            ('Плато растяжения', c.et0, c.et2, 0.0, c.Rt),
        ]
    return parts

def stress_concrete(eps: float, c: Concrete, tension=False):
    """Only call inside admissible concrete deformation range."""
    if eps >= 0 and not tension:
        return 0.0
    if eps < -c.eb2-1e-12 or (tension and eps > c.et2+1e-12):
        raise ValueError('Деформация бетона выходит за пределы выбранной диаграммы')
    for _, left, right, alpha, beta in concrete_segments(c,tension):
        if left-1e-12 <= eps <= right+1e-12:
            return alpha*eps+beta
    if not tension and eps >= 0: return 0.0
    raise ValueError(f'Недопустимая деформация: {eps}')

def stress_steel(eps: float, s:Steel):
    if abs(eps)>s.eult+1e-12:
        raise ValueError('Предельная деформация арматуры превышена')
    return min(s.E*eps,s.Rst) if eps>=0 else -min(-s.E*eps,s.Rsc)

def concrete_resultants(e0,k,c:Concrete,sec:Section=SEC,tension=False,details=False):
    """Точные N_b, M_b (Н, Н·мм) кусочно-линейной модели."""
    if abs(k*sec.h)<1e-13:
        sig=stress_concrete(e0,c,tension)
        result=(sec.b*sec.h*sig,0.0)
        if details:
            return (*result, [dict(Участок='Равномерная деформация',z1=-sec.h/2,z2=sec.h/2, sigma1=sig,sigma2=sig,dN=result[0],dM=0)])
        return result
    n,m=0.0,0.0
    contributions=[]
    for name,el,er,alpha,beta in concrete_segments(c,tension):
        za=(el-e0)/k; zb=(er-e0)/k
        z1=max(-sec.h/2, min(za,zb))
        z2=min( sec.h/2, max(za,zb))
        if z2<=z1+1e-12: continue
        sig1=alpha*(e0+k*z1)+beta
        sig2=alpha*(e0+k*z2)+beta
        dz=z2-z1
        dN=sec.b*dz*(sig1+sig2)/2
        dM=-sec.b*dz*((2*z1+z2)*sig1+(z1+2*z2)*sig2)/6
        n+=dN; m+=dM
        contributions.append(dict(Участок=name,z1=z1,z2=z2,sigma1=sig1,sigma2=sig2,dN=dN,dM=dM))
    if details: return n,m,contributions
    return n,m

def beta_concrete(k:float,sec:Section=SEC,c:Concrete=CONC_I):
    t=abs(k)*sec.h/2
    if t <= c.eb2/2:
        return t-(c.eb0+np.sqrt(c.eb0**2+8*(c.eb2-c.eb0)*t))/2
    return t-c.eb2

def beta_crack(k:float,sec:Section=SEC,c:Concrete=CONC_II):
    t=abs(k)*sec.h/2
    if t<=c.et2/2:
        return -t+(c.et0+np.sqrt(c.et0**2+8*(c.et2-c.et0)*t))/2
    return c.et2-t

def k_limits(sec:Section=SEC,c:Concrete=CONC_I,s:Steel=STEEL_I):
    return (-(c.eb2+s.eult)/(sec.h-sec.a_bot), (c.eb2+s.eult)/(sec.h-sec.a_top))

def strain_bounds(k,sec:Section=SEC,c:Concrete=CONC_I,s:Steel=STEEL_I):
    lower=max(beta_concrete(k,sec,c),-s.eult-k*sec.zb,-s.eult-k*sec.zt)
    upper=min(s.eult-k*sec.zb,s.eult-k*sec.zt)
    return lower,upper

def resultants(e0,k,As_bot,As_top,sec:Section=SEC,c:Concrete=CONC_I,s:Steel=STEEL_I,tension=False):
    nb,mb=concrete_resultants(e0,k,c,sec,tension)
    eb=e0+k*sec.zb; et=e0+k*sec.zt
    qb=stress_steel(eb,s)-stress_concrete(eb,c,tension)
    qt=stress_steel(et,s)-stress_concrete(et,c,tension)
    return nb+As_bot*qb+As_top*qt, mb-As_bot*qb*sec.zb-As_top*qt*sec.zt

def candidate_areas(e0,k,N,M,sec:Section=SEC,c:Concrete=CONC_I,s:Steel=STEEL_I):
    nb,mb=concrete_resultants(e0,k,c,sec,False)
    eb=e0+k*sec.zb; et=e0+k*sec.zt
    qb=stress_steel(eb,s)-stress_concrete(eb,c,False)
    qt=stress_steel(et,s)-stress_concrete(et,c,False)
    p=sec.zt*(N-nb)+(M-mb)
    q=-sec.zb*(N-nb)-(M-mb)
    if abs(qb)<1e-8 or abs(qt)<1e-8:
        return None  # singular cases require independent linear feasibility solver
    return p/(sec.d*qb), q/(sec.d*qt)

## 2. Расчётное ядро с Numba: две границы и первый положительный лучевой корень

Сохраняется знак моментов из исходной НДМ: `M = - ∫σ(z) z dA`. При расчёте трещинообразования применяется трёхлинейная диаграмма растянутого бетона и нормативные сопротивления. При расчёте прочности — **те же нормативные сопротивления** сжатого бетона и стали, но растянутый бетон выключен. Расчётные сопротивления I группы используются только как ключи выбора нормативных характеристик материала из таблиц.

Пересечение определяется из условия коллинеарности `(N_int, M_int)` и `(N, My)` с положительным коэффициентом λ. Граница прочности представлена нижней и верхней границами `strain_bounds`; граница трещинообразования — `beta_crack`. Для каждого участка ищется **первый положительный** корень. Внутри интервала используется бисекция без SciPy.

**Ограничение численного учителя:** сканирование по κ может пропустить касательное пересечение или дополнительную изолированную ветвь. Поэтому в разделе аудита сравниваются результаты с увеличенным числом узлов; при обнаружении сомнительных меток обучение блокируется до уточнения эталонов.
Напряжения в арматуре вычисляются **по двум отдельным равновесным состояниям**: при $\lambda_{crc}$ и при $\lambda=1$. Для обоих напряжённых состояний растянутый бетон отключён.


In [4]:
@njit(cache=False)
def rc_cstress(e, rb,eb, rt, tension):
    eb1=0.6*rb/eb
    if e < 0.0:
        if e <= -CR_EB0: return -rb
        if e <= -eb1:
            slope=0.4*rb/(CR_EB0-eb1)
            return slope*e-0.6*rb+slope*eb1
        return eb*e
    if not tension: return 0.0
    et1=0.6*rt/eb
    if e <= et1: return eb*e
    if e <= CR_ET0:
        slope=0.4*rt/(CR_ET0-et1)
        return slope*e+0.6*rt-slope*et1
    return rt

@njit(cache=False)
def rc_sstress(e,rs,rsc):
    if e >= 0.0:return min(CR_ES*e,rs)
    return -min(-CR_ES*e,rsc)

@njit(cache=False)
def rc_concrete_nm(e0,k,rb,eb,rt,h,tension):
    eb1=0.6*rb/eb
    cb=0.4*rb/(CR_EB0-eb1)
    # Одинаковые типы и длины массивов в обеих ветвях для Numba.
    et1=0.6*rt/eb if tension else CR_ET0*0.5
    ct=0.4*rt/(CR_ET0-et1) if tension else 0.0
    breaks=(-CR_EB2,-CR_EB0,-eb1,0.,et1,CR_ET0,CR_ET2)
    slopes=(0.,cb,eb,eb,ct,0.)
    intercept=(-rb,-0.6*rb+cb*eb1,0.,0.,0.6*rt-ct*et1,rt)
    count=6 if tension else 3
    half=h/2.
    if abs(k*h) < 1e-13:
        return CR_B*h*rc_cstress(e0,rb,eb,rt,tension),0.
    N=0.;M=0.
    for j in range(count):
        za=(breaks[j]-e0)/k;zb=(breaks[j+1]-e0)/k
        zl=max(-half,min(za,zb));zr=min(half,max(za,zb))
        if zr<=zl:continue
        s1=slopes[j]*(e0+k*zl)+intercept[j]
        s2=slopes[j]*(e0+k*zr)+intercept[j]
        dz=zr-zl
        N+=CR_B*dz*0.5*(s1+s2)
        M-=CR_B*dz*((2*zl+zr)*s1+(zl+2*zr)*s2)/6.
    return N,M

@njit(cache=False)
def rc_resultants(e0,k,rb,eb,rt,h,ab,at,rs,rsc,asb,ast,tension):
    N,M=rc_concrete_nm(e0,k,rb,eb,rt,h,tension)
    zb=-h/2+ab;zt=h/2-at
    eb_=e0+k*zb;et_=e0+k*zt
    qb=rc_sstress(eb_,rs,rsc)-rc_cstress(eb_,rb,eb,rt,tension)
    qt=rc_sstress(et_,rs,rsc)-rc_cstress(et_,rb,eb,rt,tension)
    return N+100.*asb*qb+100.*ast*qt, M-100.*asb*qb*zb-100.*ast*qt*zt

@njit(cache=False)
def rc_beta_concrete(k,h):
    t=abs(k)*h/2.
    if t<=CR_EB2/2.:
        return t-(CR_EB0+np.sqrt(CR_EB0*CR_EB0+8.*(CR_EB2-CR_EB0)*t))/2.
    return t-CR_EB2

@njit(cache=False)
def rc_beta_crack(k,h):
    t=abs(k)*h/2.
    if t<=CR_ET2/2.:
        return -t+(CR_ET0+np.sqrt(CR_ET0*CR_ET0+8.*(CR_ET2-CR_ET0)*t))/2.
    return CR_ET2-t

@njit(cache=False)
def rc_limits(k,rb,eb,h,ab,at):
    zb=-h/2.+ab;zt=h/2.-at
    low=max(rc_beta_concrete(k,h),-CR_ES2-k*zb,-CR_ES2-k*zt)
    high=min(CR_ES2-k*zb,CR_ES2-k*zt)
    return low,high

@njit(cache=False)
def rc_point(k,branch,mode,row,rbser,rbtser,rsser,Nex,Mex):
    # branch 0: нижняя граница прочности, 1: верхняя, 2: граница трещин
    eb=row[3];h=row[4];ab=row[5];at=row[6]
    asb=row[9];ast=row[10]
    if mode==0:
        low,high=rc_limits(k,rbser,eb,h,ab,at)
        if high < low-1e-12:return False,0.,0.
        e0=low if branch==0 else high
        n,m=rc_resultants(e0,k,rbser,eb,0.,h,ab,at,rsser,rsser,asb,ast,False)
    else:
        e0=rc_beta_crack(k,h)
        # Без трещин: соблюдаем ограничение сжатого бетона и предельные деформации стали.
        if e0 < rc_beta_concrete(k,h)-1e-11:return False,0.,0.
        zb=-h/2+ab;zt=h/2-at
        if abs(e0+k*zb)>CR_ES2+1e-11 or abs(e0+k*zt)>CR_ES2+1e-11:
            return False,0.,0.
        if e0+abs(k)*h/2. > CR_ET2+1e-10:return False,0.,0.
        n,m=rc_resultants(e0,k,rbser,eb,rbtser,h,ab,at,rsser,rsser,asb,ast,True)
    # Нормируем M на h, чтобы N и M/h были сопоставимы по масштабу.
    ni=n;mi=m/h
    det=(ni*Mex-mi*Nex)
    lam=(ni*Nex+mi*Mex)/(Nex*Nex+Mex*Mex)
    return True,det,lam

@njit(cache=False)
def rc_ray_root(ka,kb,fa,fb,branch,mode,row,rbser,rbtser,rsser,Nex,Mex,nrefine):
    # бракованные интервалы вызывающий код не передает
    left=ka;right=kb;fl=fa
    for _ in range(nrefine):
        middle=(left+right)/2.
        ok,fm,lm=rc_point(middle,branch,mode,row,rbser,rbtser,rsser,Nex,Mex)
        if not ok: return np.inf,np.nan
        if fl*fm<=0.:
            right=middle
        else:
            left=middle;fl=fm
    ok,f,lam=rc_point((left+right)/2.,branch,mode,row,rbser,rbtser,rsser,Nex,Mex)
    if ok and lam>1e-12 and abs(f)<1e-5*(1.+Nex*Nex+Mex*Mex):
        return lam,(left+right)/2.
    return np.inf,np.nan

@njit(cache=False)
def rc_boundary_min_lambda(row,rbser,rbtser,rsser,mode,nk,nrefine):
    N=row[0]*1000.;M=row[1]*1e6;h=row[4];ab=row[5];at=row[6]
    Nex=N;Mex=M/h
    if Nex*Nex+Mex*Mex < 1e-14:return np.inf,np.nan
    if mode==0:
        kl=-(CR_EB2+CR_ES2)/(h-ab)
        kr=(CR_EB2+CR_ES2)/(h-at)
        branches=2
    else:
        kl=-(CR_EB2+CR_ET2)/h
        kr= (CR_EB2+CR_ET2)/h
        branches=1
    best=np.inf
    best_k=np.nan
    for b in range(branches):
        branch=b if mode==0 else 2
        okprev=False; fprev=0.;kprev=kl
        for i in range(nk):
            k=kl+(kr-kl)*(i/(nk-1.))
            ok,f,lam=rc_point(k,branch,mode,row,rbser,rbtser,rsser,Nex,Mex)
            if ok:
                if abs(f) < 1e-9*(1+Nex*Nex+Mex*Mex) and lam>1e-12 and lam<best:
                    best=lam;best_k=k
                if okprev and f*fprev<0.:
                    root,kr=rc_ray_root(kprev,k,fprev,f,branch,mode,row,rbser,rbtser,rsser,Nex,Mex,nrefine)
                    if root<best:
                        best=root;best_k=kr
            okprev=ok;kprev=k;fprev=f
    return best,best_k



# Вычисление σ_s,crc сразу после первой трещины:
# Нагрузки берутся именно в точке λ_crc, работа растянутого бетона выключена.
@njit(cache=False)
def rc_cstress_tangent(e,rb,eb):
    if e>=0.0 or e<=-CR_EB0:return 0.0
    eb1=0.6*rb/eb
    if e<=-eb1:return 0.4*rb/(CR_EB0-eb1)
    return eb

@njit(cache=False)
def rc_sstress_tangent(e,rsser):
    if e>=rsser/CR_ES or e<=-rsser/CR_ES:return 0.0
    return CR_ES

@njit(cache=False)
def rc_concrete_tangent_k(e0,k,rb,eb,h):
    # K_j= b∫E_tan*z^j dz для j=0,1,2, бетон без растяжения.
    if abs(k*h)<1e-13:
        Et=rc_cstress_tangent(e0,rb,eb)
        return CR_B*h*Et,0.0,CR_B*h*h*h*Et/12.0
    eb1=0.6*rb/eb
    breaks=(-CR_EB2,-CR_EB0,-eb1,0.)
    slopes=(0.0,0.4*rb/(CR_EB0-eb1),eb)
    K0=0.0;K1=0.0;K2=0.0
    for i in range(3):
        za=(breaks[i]-e0)/k;zb=(breaks[i+1]-e0)/k
        lo=max(-h/2.0,min(za,zb));hi=min(h/2.0,max(za,zb))
        if hi<=lo:continue
        c=CR_B*slopes[i]
        K0+=c*(hi-lo)
        K1+=c*(hi*hi-lo*lo)/2.0
        K2+=c*(hi*hi*hi-lo*lo*lo)/3.0
    return K0,K1,K2

@njit(cache=False)
def rc_cracked_equilibrium(row, rbser, rsser, lam, e_seed, k_seed):
    """Равновесие без растянутого бетона: (sigma_sb,sigma_st,sigma_s,e0,k).

    Важно: напряжения при lambda_crc и lambda=1 решаются НЕЗАВИСИМО
    по соответствующим целевым усилиям; масштабирование sigma недопустимо.
    """
    Ntar=lam*row[0]*1000.0;Mtar=lam*row[1]*1e6
    eb=row[3];h=row[4];ab=row[5];at=row[6]
    asb=100.0*row[9];ast=100.0*row[10]
    zb=-h/2.0+ab;zt=h/2.0-at
    Ns=max(abs(Ntar),CR_B*h*rbser*0.1,1.0)
    Ms=max(abs(Mtar),Ns*h*0.5,1.0)
    best_dist=np.inf;best_e0=np.nan;best_k=np.nan
    for start in range(5):
        if start==0:
            e0=e_seed;k=k_seed
        elif start==1:
            e0=e_seed-0.00005;k=k_seed
        elif start==2:
            e0=e_seed;k=0.85*k_seed
        elif start==3:
            e0=e_seed-0.0001;k=1.10*k_seed
        else:
            e0=e_seed;k=1.15*k_seed
        for _ in range(CR_POST_NEWTON_ITERS):
            low,high=rc_limits(k,rbser,eb,h,ab,at)
            if e0<low-1e-10 or e0>high+1e-10:break
            n,m=rc_resultants(e0,k,rbser,eb,0.,h,ab,at,rsser,rsser,row[9],row[10],False)
            rn=n-Ntar;rm=m-Mtar
            err=np.sqrt((rn/Ns)**2+(rm/Ms)**2)
            if err<CR_POST_TOL:
                dist=((e0-e_seed)/0.001)**2+((k-k_seed)*h/0.001)**2
                if dist<best_dist:
                    best_dist=dist;best_e0=e0;best_k=k
                break
            K0,K1,K2=rc_concrete_tangent_k(e0,k,rbser,eb,h)
            qb=rc_sstress_tangent(e0+k*zb,rsser)-rc_cstress_tangent(e0+k*zb,rbser,eb)
            qt=rc_sstress_tangent(e0+k*zt,rsser)-rc_cstress_tangent(e0+k*zt,rbser,eb)
            a=K0+asb*qb+ast*qt
            b=K1+asb*qb*zb+ast*qt*zt
            c=-b
            d=-K2-asb*qb*zb*zb-ast*qt*zt*zt
            det=a*d-b*c
            if abs(det)<1e-16*max(1.0,abs(a*d)+abs(b*c)):break
            de=(d*rn-b*rm)/det
            dk=(-c*rn+a*rm)/det
            accepted=False;alpha=1.0
            for _ in range(22):
                etry=e0-alpha*de;ktry=k-alpha*dk
                lo,hi=rc_limits(ktry,rbser,eb,h,ab,at)
                if etry>=lo-1e-10 and etry<=hi+1e-10:
                    nt,mt=rc_resultants(etry,ktry,rbser,eb,0.,h,ab,at,
                                        rsser,rsser,row[9],row[10],False)
                    rt=np.sqrt(((nt-Ntar)/Ns)**2+((mt-Mtar)/Ms)**2)
                    if rt<err:
                        e0=etry;k=ktry;accepted=True;break
                alpha*=0.5
            if not accepted:break
    if not np.isfinite(best_e0):
        return np.nan,np.nan,np.nan,np.nan,np.nan
    sb=rc_sstress(best_e0+best_k*zb,rsser,rsser)
    st=rc_sstress(best_e0+best_k*zt,rsser,rsser)
    if sb<=0.0 and st<=0.0:
        return np.nan,np.nan,np.nan,np.nan,np.nan
    return sb,st,max(0.0,sb,st),best_e0,best_k

@njit(cache=False)
def rc_postcrack_equilibrium(row,rbser,rsser,lam,kpre):
    """Совместимость с прежним API, три значения при заданном lambda."""
    sb,st,sig,e0,k=rc_cracked_equilibrium(
        row,rbser,rsser,lam,rc_beta_crack(kpre,row[4]),kpre)
    return sb,st,sig

@njit(cache=False)
def rc_full_load_stresses(row,rbser,rsser,lam_crc, e_crack,k_crack):
    """Растянутый бетон выключен; полное N,My соответствует lambda=1."""
    sb,st,_,_,_=rc_cracked_equilibrium(row,rbser,rsser,1.,e_crack,k_crack)
    if np.isfinite(sb) and np.isfinite(st):return sb,st
    # При большой разнице lambda используем продолжение ветви равновесия.
    e=e_crack;k=k_crack
    for j in range(1,7):
        l=lam_crc+(1.-lam_crc)*j/6.
        sj,tj,_,ee,kk=rc_cracked_equilibrium(row,rbser,rsser,l,e,k)
        if not np.isfinite(ee):return np.nan,np.nan
        e=ee;k=kk
    return sj,tj

@njit(cache=False)
def rc_teacher(row,rbser,rbtser,rsser,nkult,nkcrc,nrefine,tol):
    # Цели: lambda_first, lambda_ult, lambda_crc_if_first, event,
    #        sigma_sb_crc, sigma_st_crc, sigma_sb_full, sigma_st_full.
    lu,_=rc_boundary_min_lambda(row,rbser,rbtser,rsser,0,nkult,nrefine)
    if not np.isfinite(lu):
        return np.nan,np.nan,np.nan,-1,np.nan,np.nan,np.nan,np.nan
    lc,kc=rc_boundary_min_lambda(row,rbser,rbtser,rsser,1,nkcrc,nrefine)
    if not np.isfinite(lc) or lc>lu*(1.+tol) or not np.isfinite(kc):
        return lu,lu,np.nan,0,np.nan,np.nan,np.nan,np.nan
    sb,st,_,ecr,kcr=rc_cracked_equilibrium(
        row,rbser,rsser,lc,rc_beta_crack(kc,row[4]),kc)
    # Sigma_full имеет смысл только после первой трещины и до предела прочности.
    sf_b=np.nan;sf_t=np.nan
    if np.isfinite(ecr) and lc<=1.+tol and lu>=1.-tol:
        sf_b,sf_t=rc_full_load_stresses(row,rbser,rsser,lc,ecr,kcr)
    if abs(lc-lu)<=tol*max(1e-8,lu):
        return min(lu,lc),lu,lc,2,sb,st,sf_b,sf_t
    return lc,lu,lc,1,sb,st,sf_b,sf_t

@njit(parallel=True,cache=False)
def rc_teacher_batch(X,C,S,nkult,nkcrc,nrefine,tol):
    n=len(X)
    first=np.empty(n);ult=np.empty(n);crc=np.empty(n)
    event=np.empty(n,dtype=np.int64)
    sigb=np.empty(n);sigt=np.empty(n)
    sigb_full=np.empty(n);sigt_full=np.empty(n)
    for i in prange(n):
        f,u,c,e,sb,st,sbf,stf=rc_teacher(X[i],C[i,0],C[i,1],S[i],
                                            nkult,nkcrc,nrefine,tol)
        first[i]=f;ult[i]=u;crc[i]=c;event[i]=e
        sigb[i]=sb;sigt[i]=st
        sigb_full[i]=sbf;sigt_full[i]=stf
    return first,ult,crc,event,sigb,sigt,sigb_full,sigt_full


## 3. Генерация собственной выборки

Площади `A_sb` и `A_st` — **входные фактические площади**, а не метки подбора. Никакого увеличения нагрузок или площадей здесь нет. Прочностное и трещиностойкое состояния рассчитываются для этих же площадей, оба — по нормативным характеристикам материалов.

Для лучей используются разнообразные исходные деформационные состояния: центральное сжатие, центральное растяжение, изгиб, состояния вблизи границы трещин и комбинированные напряжённые состояния. Метки, для которых не найден конечный положительный индекс предела прочности либо индекс выходит за заданный диапазон, отбрасываются; это **фильтр области обучения**, а не утверждение о невозможности состояния.
Привязки $a_{sb}$, $a_{st}$ генерируются отдельно в интервале $[a_{min},\,0{,}45h]$.

**Эталонные напряжения на трещине:** для события `CRACK_FIRST` / `SIMULTANEOUS` вычисляются `sigma_sb_crc` и `sigma_st_crc` (МПа) сразу после трещинообразования, при `N_crc, M_crc`, из НДМ **без растянутого бетона**. Отрицательные значения означают сжатие арматуры. Если предельная прочность достигнута раньше или равновесие после выключения растянутого бетона не найдено, значения `NaN`. При генерации метки события `CRACK_FIRST` с неопределёнными напряжениями не включаются в обучение. Это не формула ширины раскрытия, а один из её исходных параметров.

**Эталонные напряжения при полной нагрузке:** `sigma_sb_full` и `sigma_st_full` (МПа) определяются при исходных `N, My`, то есть $\lambda=1$, **без растянутого бетона**. Они имеют физический смысл для образовавшихся трещин, если $\lambda_{crc}\le1$ и $\lambda_{ult}\ge1$. В остальных случаях или при неудаче решения равновесия сохраняется `NaN`. Напряжения на трещине и при полной нагрузке не отождествляются.


In [5]:
CR_INPUT_COLUMNS = ('N_kN_m','My_kNm_m','Rb_MPa','Eb_MPa','h_mm','asb_mm','ast_mm',
                    'Rs_MPa','Rsc_MPa','Asb_cm2_m','Ast_cm2_m')
CR_EVENTS = ('STRENGTH_FIRST','CRACK_FIRST','SIMULTANEOUS')
CR_TARGET_COLUMNS = ('lambda_first','lambda_ult','lambda_crc_if_first',
                     'event','sigma_sb_crc_MPa','sigma_st_crc_MPa',
                     'sigma_sb_full_MPa','sigma_st_full_MPa')

def rc_area_bounds(h):
    lo=max(CR_A_MIN,CR_RHO_MIN_PCT*h/10.)
    hi=min(CR_A_MAX,CR_RHO_MAX_PCT*h/10.)
    return lo,hi

def rc_material_lookup(rb,rs,rsc):
    cc=[r for r in CR_CONCRETE_MATRIX if abs(r[1]-rb)<1e-5]
    ss=[r for r in CR_STEEL_MATRIX if abs(r[1]-rs)<1e-5 and abs(r[2]-rsc)<1e-5]
    if len(cc)!=1 or len(ss)!=1:
        raise ValueError('Rb или (Rs,Rsc) отсутствуют в матрицах. Добавь соответствующую строку.')
    return cc[0][3],cc[0][4],ss[0][3]

def rc_generate_inputs(count,seed):
    if not (0 <= CR_AS_OFFSET_MIN < CR_H_MIN*CR_AS_OFFSET_MAX_H_FRAC
            and 0 < CR_AS_OFFSET_MAX_H_FRAC < 0.5):
        raise ValueError('Некорректные привязки: 0 <= min_mm < frac*h_min и 0 < frac < 0.5.')
    rng=np.random.default_rng(seed)
    X=np.empty((count,11),dtype=np.float64)
    C=np.empty((count,2),dtype=np.float64)
    S=np.empty(count,dtype=np.float64)
    for i in range(count):
        bc=CR_CONCRETE_MATRIX[int(rng.integers(len(CR_CONCRETE_MATRIX)))]
        st=CR_STEEL_MATRIX[int(rng.integers(len(CR_STEEL_MATRIX)))]
        _,rb,eb,rbser,rbtser=bc
        _,rs,rsc,rsser=st
        h=rng.uniform(CR_H_MIN,CR_H_MAX)
        ab=rng.uniform(CR_AS_OFFSET_MIN, CR_AS_OFFSET_MAX_H_FRAC*h)
        at=rng.uniform(CR_AS_OFFSET_MIN, CR_AS_OFFSET_MAX_H_FRAC*h)
        amin,amax=rc_area_bounds(h)
        if amin>=amax:raise ValueError('Пустой диапазон процентов/площадей')
        asb=np.exp(rng.uniform(np.log(amin),np.log(amax)))
        ast=np.exp(rng.uniform(np.log(amin),np.log(amax)))
        mode=i%6
        if mode==0:  # преимущественно сжатие
            k=rng.uniform(-3e-6,3e-6)*(200./h)
            e0=-rng.uniform(0.00012,0.0017)
            mat=(rbser,0.,rsser,rsser,False)
        elif mode==1: # преимущественно растяжение
            k=rng.uniform(-2e-7,2e-7)*(200./h)
            e0=rng.uniform(0.000015,CR_ET0*0.9)
            mat=(rbser,rbtser,rsser,rsser,True)
        elif mode==2: # на границе трещин, знак изгиба произвольный
            k=rng.uniform(-0.95,0.95)*((CR_EB2+CR_ET2)/h)
            e0=rc_beta_crack(k,h)
            if e0<rc_beta_concrete(k,h): k*=0.5;e0=rc_beta_crack(k,h)
            mat=(rbser,rbtser,rsser,rsser,True)
        else: # основная область деформаций по прочности
            kl=-(CR_EB2+CR_ES2)/(h-ab)
            kr=(CR_EB2+CR_ES2)/(h-at)
            k=rng.uniform(0.95*kl,0.95*kr)
            low,high=rc_limits(k,rbser,eb,h,ab,at)
            if high<low:
                for _ in range(35):
                    k*=0.85;low,high=rc_limits(k,rbser,eb,h,ab,at)
                    if high>=low:break
            e0=low+rng.uniform(0.02,0.98)*(high-low)
            mat=(rbser,0.,rsser,rsser,False)
        n,m=rc_resultants(e0,k,mat[0],eb,mat[1],h,ab,at,mat[2],mat[3],asb,ast,mat[4])
        # Масштабирование НАГРУЗКИ нужно только для генерации разнообразных лучей;
        # эталон определяется по исходным характеристикам и не содержит искусственного запаса.
        fac=np.exp(rng.uniform(np.log(0.40),np.log(2.0)))
        X[i]=[fac*n/1000.,fac*m/1e6,rb,eb,h,ab,at,rs,rsc,asb,ast]
        C[i]=[rbser,rbtser]
        S[i]=rsser
    return X,C,S

def rc_make_dataset(n=CR_N_SAMPLES,seed=CR_SEED,batch=CR_BATCH):
    rng=np.random.default_rng(seed)
    XX=[];YY=[];CC=[];SS=[];total=0;tries=0
    start=time.perf_counter()
    while total<n:
        tries+=1
        if tries>n//batch*20+200:
            raise RuntimeError('Слишком мала доля пригодных эталонов: проверь ограничения и поиск корней')
        size=min(batch,max(n-total,batch))
        X,C,S=rc_generate_inputs(size,int(rng.integers(2**31-1)))
        first,ult,crc,event,sigb,sigt,sigb_full,sigt_full=rc_teacher_batch(
            X,C,S,CR_K_POINTS_ULT,CR_K_POINTS_CRC,CR_REFINE_ITERS,CR_SIMULTANEOUS_REL_TOL)
        valid=(event>=0)&np.isfinite(first)&np.isfinite(ult)&(first>=CR_LAMBDA_MIN)&(first<=CR_LAMBDA_MAX)&(ult<=CR_LAMBDA_MAX)
        valid &= ((event==0)|(np.isfinite(sigb)&np.isfinite(sigt)))
        # Для случаев, когда при полной нагрузке трещина уже есть и прочность
        # ещё не исчерпана, требуем оба надёжно вычисленных эталона sigma_full.
        full_needed=(event!=0)&(crc<=1.+CR_SIMULTANEOUS_REL_TOL)&(ult>=1.-CR_SIMULTANEOUS_REL_TOL)
        valid &= (~full_needed)|(np.isfinite(sigb_full)&np.isfinite(sigt_full))
        idx=np.flatnonzero(valid)[:n-total]
        if len(idx):
            XX.append(X[idx]);CC.append(C[idx]);SS.append(S[idx]);
            YY.append(np.column_stack([first[idx],ult[idx],crc[idx],event[idx],
                                          sigb[idx],sigt[idx],sigb_full[idx],sigt_full[idx]]));total+=len(idx)
        if tries%5==0 or total==n:
            print(f'Лучевые эталоны: {total}/{n} | принято {sum(valid)}/{size} | {time.perf_counter()-start:.1f} с',flush=True)
    X=np.vstack(XX);Y=np.vstack(YY);C=np.vstack(CC);S=np.concatenate(SS)
    np.savez_compressed(CR_DATA_FILE,X=X,Y=Y,C=C,S=S,
                        input_columns=np.array(CR_INPUT_COLUMNS),target_columns=np.array(CR_TARGET_COLUMNS),events=np.array(CR_EVENTS),
                        concrete_matrix=np.asarray(CR_CONCRETE_MATRIX,dtype='U32'),
                        steel_matrix=np.asarray(CR_STEEL_MATRIX,dtype='U32'),
                        seed=seed,beam_width_mm=CR_B,
                        as_offset_min_mm=np.float64(CR_AS_OFFSET_MIN),
                        as_offset_max_h_frac=np.float64(CR_AS_OFFSET_MAX_H_FRAC),
                        nkult=CR_K_POINTS_ULT,nkcrc=CR_K_POINTS_CRC,
                        strength_material_basis='Rb_ser/Rs_ser, concrete_tension=False',
                        crack_material_basis='Rb_ser/Rbt_ser/Rs_ser, concrete_tension=True',
                        dataset_version='crack-ray-v4-normative-crack-and-full-stress')
    print('Выборка сохранена:',CR_DATA_FILE, '|',len(X),'записей')
    return X,Y,C,S

def rc_audit(X,Y,C,S,n=CR_AUDIT_COUNT,seed=42):
    rng=np.random.default_rng(seed)
    idx=rng.choice(len(X),size=min(n,len(X)),replace=False)
    f,u,c,e,sb,st,sbf,stf=rc_teacher_batch(X[idx],C[idx],S[idx],
                               CR_K_POINTS_ULT*2-1,CR_K_POINTS_CRC*2-1,
                               CR_REFINE_ITERS,CR_SIMULTANEOUS_REL_TOL)
    good=(e==Y[idx,3]) & np.isfinite(f) & np.isfinite(u)
    good &= (abs(f-Y[idx,0]) <= .005*np.maximum(1.,Y[idx,0]))
    good &= (abs(u-Y[idx,1]) <= .005*np.maximum(1.,Y[idx,1]))
    # Дополнительная проверка σ после трещины; сжатие и растяжение со знаком.
    stress_mask=(e!=0)&np.isfinite(Y[idx,4])&np.isfinite(sb)&np.isfinite(st)
    stress_expected=(Y[idx,3]!=0)
    stress_ok=np.ones(len(idx),dtype=bool)
    if np.any(stress_expected):
        # Переходы по различным лучевым корням могут менять напряжения скачком.
        stress_ok[stress_expected]=False
        if np.any(stress_mask):
            d=np.maximum(np.abs(sb[stress_mask]-Y[idx[stress_mask],4]),
                         np.abs(st[stress_mask]-Y[idx[stress_mask],5]))
            limit=np.maximum(2.0,0.02*S[idx[stress_mask]])
            stress_ok[stress_mask]=d<=limit
            print('Аудит σ_s,crc, МПа: max |Δσ| =',float(np.max(d)),
                  '| совпало:',int(np.sum(d<=limit)),'/',len(d))
    good &= stress_ok
    # Два дополнительных напряжения при исходной полной нагрузке (lambda=1).
    full_expected=(Y[idx,3]!=0) & (Y[idx,2]<=1.+CR_SIMULTANEOUS_REL_TOL) & (Y[idx,1]>=1.-CR_SIMULTANEOUS_REL_TOL)
    full_mask=full_expected & np.isfinite(Y[idx,6]) & np.isfinite(Y[idx,7]) & np.isfinite(sbf) & np.isfinite(stf)
    full_ok=np.ones(len(idx),dtype=bool)
    if np.any(full_expected):
        full_ok[full_expected]=False
        if np.any(full_mask):
            d_full=np.maximum(np.abs(sbf[full_mask]-Y[idx[full_mask],6]),
                              np.abs(stf[full_mask]-Y[idx[full_mask],7]))
            limit_full=np.maximum(2.,0.02*S[idx[full_mask]])
            full_ok[full_mask]=d_full<=limit_full
            print('Аудит sigma_s,full, МПа: max |delta| =',float(np.max(d_full)),
                  '| совпало:',int(np.sum(d_full<=limit_full)),'/',len(d_full))
    good &= full_ok
    print(f'Аудит повышенной плотности: {sum(good)}/{len(idx)} совпадений (0.5% λ, статус, σ_s,crc и σ_s,full)')
    if np.mean(good)<0.995:
        print('ВНИМАНИЕ: обнаружены нестабильные метки. Рекомендуется увеличить CR_K_POINTS_* и пересобрать датасет.')
    return good

## 4. Подготовка признаков и обучение компактной сети с напряжениями σ_s,crc и σ_s,full (без SciPy)


In [6]:
# 22 инженерных признака от 11 физических входов

def rc_features(X):
    X=np.atleast_2d(np.asarray(X,dtype=np.float64))
    n,m,rb,eb,h,ab,at,rs,rsc,asb,ast=X.T
    rbser=np.empty(len(X));rbtser=np.empty(len(X));rsser=np.empty(len(X))
    for j,(r,s,t) in enumerate(zip(rb,rs,rsc)):
        rbser[j],rbtser[j],rsser[j]=rc_material_lookup(r,s,t)
    nn=n*1000./(CR_B*h*rb)
    mm=m*1e6/(CR_B*h*h*rb)
    return np.column_stack((
        nn,mm,np.abs(nn),np.abs(mm),nn*mm,
        rb/35.,eb/40000.,h/300.,ab/h,at/h,
        rs/500.,rsc/500.,asb/(h/10.),ast/(h/10.),
        asb/40.,ast/40.,CR_ES/eb,
        rbser/rb,rbtser/rb,rsser/500.,
        nn/(1.+np.abs(mm)),mm/(1.+np.abs(nn))
    )).astype(np.float32)

class CrackRayNet(nn.Module):
    def __init__(self,dim=22,width=CR_HIDDEN):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(dim,width),nn.SiLU(),
            nn.Linear(width,width),nn.SiLU(),nn.Linear(width,width),nn.SiLU(),
            nn.Linear(width,9))
    def forward(self,x):return self.net(x)

def rc_split(n,seed=CR_SEED):
    r=np.random.default_rng(seed).permutation(n)
    ntr=int(CR_TRAIN_PART*n);nv=int(CR_VAL_PART*n)
    return r[:ntr],r[ntr:ntr+nv],r[ntr+nv:]

def rc_sigma_loss(pred, sigma_target, sigma_valid):
    """Независимая маскированная регрессия напряжений при lambda_crc и 1."""
    sigma_pred=torch.tanh(pred[:,2:6])
    loss=sigma_pred.new_zeros(())
    for lo in (0,2):
        mask=sigma_valid[:,lo:lo+2]
        if bool(mask.any()):
            err=nn.functional.smooth_l1_loss(sigma_pred[:,lo:lo+2],
                                             sigma_target[:,lo:lo+2],
                                             beta=.10,reduction='none')
            loss=loss+err[mask].mean()
    return loss
def rc_train(X,Y):
    torch.manual_seed(CR_SEED)
    features=rc_features(X)
    if features.shape[1]!=22:raise RuntimeError('Неверное число признаков')
    tr,va,te=rc_split(len(X))
    xmean=features[tr].mean(axis=0)
    xstd=np.maximum(features[tr].std(axis=0),1e-6)
    Z=((features-xmean)/xstd).astype(np.float32)
    # Обе регрессионные цели конечны. lambda_crc не нужна в отказных случаях.
    T=np.log(np.maximum(Y[:,:2],1e-9)).astype(np.float32)
    ymean=T[tr].mean(axis=0)
    ystd=np.maximum(T[tr].std(axis=0),1e-6)
    T=(T-ymean)/ystd
    # Нормированные нормативные напряжения по слоям, только в строках с трещиной.
    # На обоих уровнях нагрузки два напряжения по слоям; NaN маскируются.
    rsser_per_row=np.asarray([rc_material_lookup(*X[i,[2,7,8]])[2] for i in range(len(X))])
    sigma_valid=np.isfinite(Y[:,4:8])
    sigma=np.nan_to_num(Y[:,4:8]/rsser_per_row[:,None], nan=0.)
    sigma=np.clip(sigma,-1.,1.).astype(np.float32)
    events=Y[:,3].astype(np.int64)
    model=CrackRayNet()
    counts=np.bincount(events[tr],minlength=3).astype(np.float64)
    class_weights=np.sqrt(counts.max()/np.maximum(counts,1.))
    class_weights=np.minimum(class_weights,10.)
    class_weights=torch.tensor(class_weights,dtype=torch.float32)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    class_weights=class_weights.to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=CR_LR,weight_decay=1e-5)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='min',factor=.5,patience=9)
    zt=torch.from_numpy(Z);tt=torch.from_numpy(T);et=torch.from_numpy(events)
    st=torch.from_numpy(sigma);sv=torch.from_numpy(sigma_valid)
    from torch.utils.data import TensorDataset,DataLoader
    def loader(ids,shuffle):
        return DataLoader(TensorDataset(zt[ids],tt[ids],et[ids],st[ids],sv[ids]),batch_size=CR_BATCH_SIZE,shuffle=shuffle)
    ltr=loader(tr,True);lva=loader(va,False)
    best=np.inf;best_state=None;pat=0;start=time.perf_counter()
    for epoch in range(1,CR_EPOCHS+1):
        model.train();tl=0.;tn=0
        for bx,by,bc,bs,bv in ltr:
            bx=bx.to(device);by=by.to(device);bc=bc.to(device)
            bs=bs.to(device);bv=bv.to(device)
            opt.zero_grad(set_to_none=True)
            pred=model(bx)
            loss=nn.functional.smooth_l1_loss(pred[:,:2],by,beta=.15)+CR_WEIGHT_EVENTS*nn.functional.cross_entropy(pred[:,6:],bc,weight=class_weights)
            loss=loss+CR_WEIGHT_SIGMA*rc_sigma_loss(pred,bs,bv)
            loss.backward();opt.step()
            tl+=loss.item()*len(bx);tn+=len(bx)
        model.eval();vl=0.;vn=0
        with torch.no_grad():
            for bx,by,bc,bs,bv in lva:
                pred=model(bx.to(device));by=by.to(device);bc=bc.to(device)
                bs=bs.to(device);bv=bv.to(device)
                loss=nn.functional.smooth_l1_loss(pred[:,:2],by,beta=.15)+CR_WEIGHT_EVENTS*nn.functional.cross_entropy(pred[:,6:],bc,weight=class_weights)
                loss=loss+CR_WEIGHT_SIGMA*rc_sigma_loss(pred,bs,bv)
                vl+=loss.item()*len(bx);vn+=len(bx)
        val=vl/vn;scheduler.step(val)
        if val<best-1e-6:
            best=val;best_state={k:v.cpu().clone() for k,v in model.state_dict().items()};pat=0
        else:pat+=1
        if epoch==1 or epoch%10==0 or pat>=CR_PATIENCE:
            print(f'epoch {epoch:3d}/{CR_EPOCHS}: train={tl/tn:.6f}, val={val:.6f}, best={best:.6f}, {time.perf_counter()-start:.1f} с')
        if pat>=CR_PATIENCE:break
    model.load_state_dict(best_state);model.cpu().eval()
    layers=[v for v in model.net if isinstance(v,nn.Linear)]
    data={'x_mean':xmean.astype(np.float32),'x_std':xstd.astype(np.float32),
          'y_mean':ymean.astype(np.float32),'y_std':ystd.astype(np.float32),
          'architecture':np.array([22,CR_HIDDEN,CR_HIDDEN,CR_HIDDEN,9],dtype=np.int32),
          'input_columns':np.asarray(CR_INPUT_COLUMNS),
          'concrete_matrix':np.asarray(CR_CONCRETE_MATRIX,dtype='U32'),
          'steel_matrix':np.asarray(CR_STEEL_MATRIX,dtype='U32'),
          'event_names':np.asarray(CR_EVENTS),
          'as_offset_min_mm':np.array(CR_AS_OFFSET_MIN,dtype=np.float32),
          'as_offset_max_h_frac':np.array(CR_AS_OFFSET_MAX_H_FRAC,dtype=np.float32),
          'model_version':np.array('crack-ray-v4-normative-crack-and-full-stress'),
          'strength_material_basis':np.array('Rb_ser/Rs_ser, concrete_tension=False'),
          'crack_material_basis':np.array('Rb_ser/Rbt_ser/Rs_ser, concrete_tension=True'),
          'stress_material_basis':np.array('Rb_ser/Rs_ser, concrete_tension=False; at lambda_crc and lambda=1'),
          'sigma_outputs':np.array(['sigma_sb_crc_MPa','sigma_st_crc_MPa',
                                      'sigma_sb_full_MPa','sigma_st_full_MPa']),
          'target_columns':np.array(CR_TARGET_COLUMNS)}
    for i,layer in enumerate(layers):
        data[f'W{i}']=layer.weight.detach().numpy().T.copy().astype(np.float32)
        data[f'b{i}']=layer.bias.detach().numpy().astype(np.float32)
    np.savez_compressed(CR_MODEL_FILE,**data)
    count=sum(w.numel() for w in model.parameters())
    print('Сохранена модель:',CR_MODEL_FILE,'|',count,'float32 параметров |',count*4/1024,'КиБ весов')
    return model,te

def rc_silu(x):
    return x/(1.+np.exp(-np.clip(x,-60.,60.)))

def rc_predict(X,path=CR_MODEL_FILE):
    X=np.atleast_2d(np.asarray(X,dtype=np.float64))
    with np.load(path) as d:
        ver = str(d['model_version'].item()) if 'model_version' in d else ''
        if ver != 'crack-ray-v4-normative-crack-and-full-stress':
            raise ValueError('Несовместимые веса: требуется обучение модели crack-ray-v4-normative-crack-and-full-stress')
        feat=rc_features(X)
        x=(feat-d['x_mean'])/d['x_std']
        for i in range(4):
            x=x@d[f'W{i}']+d[f'b{i}']
            if i<3:x=rc_silu(x)
        limits=np.exp(np.clip(x[:,:2]*d['y_std']+d['y_mean'],-15.,15.))
        sig_normalized=np.tanh(x[:,2:6])
        logits=x[:,6:]-x[:,6:].max(axis=1,keepdims=True)
        prob=np.exp(logits);prob/=prob.sum(axis=1,keepdims=True)
    # В силу ошибок нейросети согласуем границы postprocess:
    ult=limits[:,1]
    event=np.argmax(prob,axis=1)
    # Непротиворечивая выдача: если первым достигается прочность, то lambda_first=lambda_ult.
    first=np.where(event==1,np.minimum(limits[:,0],ult),ult)
    has_crack=(event==1)|(event==2)
    crack=np.where(has_crack,first,np.nan)
    rsser=np.array([rc_material_lookup(*row[[2,7,8]])[2] for row in X])
    sigma_pred=sig_normalized*rsser[:,None]
    sig_layers=np.where(has_crack[:,None],sigma_pred[:,0:2],np.nan)
    sig_crc=np.where(has_crack,np.maximum(0.,np.maximum(sig_layers[:,0],sig_layers[:,1])),np.nan)
    # Напряжения от полной нагрузки относятся только к трещиноватому,
    # но ещё не разрушившемуся сечению.
    cracked_at_full=has_crack & (first<=1.+CR_SIMULTANEOUS_REL_TOL) & (ult>=1.-CR_SIMULTANEOUS_REL_TOL)
    sig_full_layers=np.where(cracked_at_full[:,None],sigma_pred[:,2:4],np.nan)
    sig_full=np.where(cracked_at_full,
                      np.maximum(0.,np.maximum(sig_full_layers[:,0],sig_full_layers[:,1])),np.nan)
    return {'lambda_first':first,'lambda_ult':ult,
            'lambda_crc_if_first':crack,'event':event,'probabilities':prob,
            'sigma_sb_crc':sig_layers[:,0],'sigma_st_crc':sig_layers[:,1],
            'sigma_s_crc':sig_crc,
            'sigma_sb_full':sig_full_layers[:,0],'sigma_st_full':sig_full_layers[:,1],
            'sigma_s_full':sig_full,'sigma_full_defined':cracked_at_full}

## 5. Генерация новой выборки и обучение с нуля

Запусти этот блок только при готовности формировать весь датасет. **Предыдущая ML-модель подбора армирования не загружается и не изменяется**. Это самостоятельный эксперимент. Перед обучением выполняется выборочная проверка независимым уточнением по более плотной сетке. При частых расхождениях увеличь `CR_K_POINTS_*` и пересобери выборку.
**Обратите внимание:** добавление напряжений от полной нагрузки изменило состав `Y` и архитектуру сети. Все прежние веса `crack-ray-v2` и `crack-ray-v3` несовместимы. Создаётся новый датасет и новая модель (`...sigma_crc_full.npz`).


In [7]:
X,Y,C,S=rc_make_dataset()
print('Формы:',X.shape,Y.shape,'События:',dict(zip(*np.unique(Y[:,3].astype(int),return_counts=True))))
_rc_audit_flags=rc_audit(X,Y,C,S)
if not np.all(_rc_audit_flags):
    raise RuntimeError('Найдено расхождение меток при сгущении сетки. Увеличь CR_K_POINTS_* и перегенерируй датасет перед обучением.')

Лучевые эталоны: 2955/300000 | принято 609/2048 | 7.6 с
Лучевые эталоны: 5965/300000 | принято 572/2048 | 7.9 с
Лучевые эталоны: 8902/300000 | принято 587/2048 | 8.2 с
Лучевые эталоны: 11889/300000 | принято 600/2048 | 8.5 с
Лучевые эталоны: 14891/300000 | принято 591/2048 | 8.8 с
Лучевые эталоны: 17830/300000 | принято 593/2048 | 9.1 с
Лучевые эталоны: 20759/300000 | принято 599/2048 | 9.4 с
Лучевые эталоны: 23706/300000 | принято 598/2048 | 9.7 с
Лучевые эталоны: 26681/300000 | принято 628/2048 | 10.1 с
Лучевые эталоны: 29683/300000 | принято 613/2048 | 10.4 с
Лучевые эталоны: 32670/300000 | принято 581/2048 | 10.7 с
Лучевые эталоны: 35639/300000 | принято 595/2048 | 11.0 с
Лучевые эталоны: 38625/300000 | принято 586/2048 | 11.3 с
Лучевые эталоны: 41658/300000 | принято 612/2048 | 11.6 с
Лучевые эталоны: 44695/300000 | принято 581/2048 | 11.9 с
Лучевые эталоны: 47692/300000 | принято 583/2048 | 12.2 с
Лучевые эталоны: 50686/300000 | принято 614/2048 | 12.5 с
Лучевые эталоны: 53628/30

In [8]:
model,test_indices=rc_train(X,Y)

epoch   1/200: train=0.367103, val=0.203259, best=0.203259, 4.5 с
epoch  10/200: train=0.061316, val=0.057762, best=0.057762, 46.6 с
epoch  20/200: train=0.042718, val=0.043484, best=0.043484, 92.8 с
epoch  30/200: train=0.034466, val=0.045070, best=0.036311, 139.2 с
epoch  40/200: train=0.029788, val=0.031094, best=0.029646, 186.1 с
epoch  50/200: train=0.026729, val=0.030066, best=0.027187, 233.8 с
epoch  60/200: train=0.025432, val=0.028882, best=0.024176, 280.0 с
epoch  70/200: train=0.024077, val=0.025580, best=0.022931, 326.6 с
epoch  80/200: train=0.023137, val=0.021950, best=0.019607, 373.3 с
epoch  90/200: train=0.015752, val=0.019095, best=0.019095, 419.8 с
epoch 100/200: train=0.015877, val=0.019780, best=0.017575, 466.8 с
epoch 110/200: train=0.013048, val=0.017418, best=0.015510, 515.6 с
epoch 120/200: train=0.012618, val=0.016012, best=0.015383, 564.3 с
epoch 130/200: train=0.011065, val=0.014752, best=0.014630, 612.6 с
epoch 140/200: train=0.010904, val=0.014491, best=0.

## 6. Независимая оценка (без подмены эталонных коэффициентов)

Основные метрики: абсолютная/относительная ошибка обоих индексов, точность классификации, ошибки **около λ=1** и доля ошибочно благоприятных прогнозов, когда истинный предел уже достигнут. Условная `lambda_crc_if_first` не означает расчёт полной кривой трещинообразования в случаях без предшествующих трещин.
Дополнительно рассчитываются ошибки напряжений после трещины и при полной нагрузке только по валидным эталонным значениям, отдельно выводится доля прогнозов с определённым $\sigma_{s,full}$.


In [9]:
def rc_report(X,Y,test_indices):
    p=rc_predict(X[test_indices]);y=Y[test_indices];eps=1e-12
    for col,key in [(0,'lambda_first'),(1,'lambda_ult')]:
        error=np.abs(p[key]-y[:,col]);rel=error/np.maximum(y[:,col],eps)
        print(f'{key}: MAE={error.mean():.5g}, MAPE={100*rel.mean():.3f}%, p95_rel={100*np.quantile(rel,.95):.3f}%')
    mask=np.isfinite(y[:,4])&np.isfinite(y[:,5])
    for col,key in [(4,'sigma_sb_crc'),(5,'sigma_st_crc'),
                    (6,'sigma_sb_full'),(7,'sigma_st_full')]:
        target_mask=np.isfinite(y[:,col])
        covered=target_mask & np.isfinite(p[key])
        print(f'{key}: меток={int(np.sum(target_mask))}, прогнозов={int(np.sum(covered))}')
        if np.any(covered):
            delta=np.abs(p[key][covered]-y[covered,col])
            print(f'  MAE={delta.mean():.4f} МПа, p95={np.quantile(delta,.95):.4f} МПа')
    print('sigma_full определена ML для',int(np.sum(p['sigma_full_defined'])),
          'из',len(y),'примеров')
    event=y[:,3].astype(int)
    print('Точность типа события:',np.mean(event==p['event']))
    print('Матрица ошибок, строки=эталон; столбцы=прогноз:')
    print(np.array([[int(np.sum((event==i)&(p['event']==j))) for j in range(3)] for i in range(3)]))
    for name,key,col in [('первое событие','lambda_first',0),('предел прочности','lambda_ult',1)]:
        false_safe=np.mean((p[key]>=1.) & (y[:,col]<1.))
        near=np.abs(y[:,col]-1.)<0.10
        print(name,': false-safe на всех тестовых:',f'{false_safe*100:.3f}%',
              '| MAE в зоне ±10%:',np.mean(np.abs(p[key][near]-y[near,col])) if np.any(near) else 'нет точек')
    return p

results=rc_report(X,Y,test_indices)

lambda_first: MAE=0.023773, MAPE=1.345%, p95_rel=4.506%
lambda_ult: MAE=0.0321, MAPE=1.197%, p95_rel=4.185%
sigma_sb_crc: меток=6609, прогнозов=6587
  MAE=6.7171 МПа, p95=22.3740 МПа
sigma_st_crc: меток=6609, прогнозов=6587
  MAE=8.3872 МПа, p95=29.5244 МПа
sigma_sb_full: меток=2197, прогнозов=2140
  MAE=6.0653 МПа, p95=19.0052 МПа
sigma_st_full: меток=2197, прогнозов=2140
  MAE=10.1653 МПа, p95=34.9483 МПа
sigma_full определена ML для 2202 из 30000 примеров
Точность типа события: 0.9969
Матрица ошибок, строки=эталон; столбцы=прогноз:
[[23338    37    16]
 [   19  6567     8]
 [    3    10     2]]
первое событие : false-safe на всех тестовых: 0.387% | MAE в зоне ±10%: 0.011154530167837582
предел прочности : false-safe на всех тестовых: 0.317% | MAE в зоне ±10%: 0.009716275665913694


## 7. Пример: один набор данных — λ, σ_s,crc, σ_s,full и точная проверка НДМ


In [10]:
# 11 входных значений: [N,My,Rb,Eb,h,asb,ast,Rs,Rsc,Asb,Ast]
# N — кН/м, My — кН·м/м, R/E — МПа, h/a — мм, A — см²/м
sample=np.array([[-168.,35.,17.,32500.,200.,30.,30.,435.,400.,8.,8.]],dtype=np.float64)
rbser,rbtser,rsser=rc_material_lookup(sample[0,2],sample[0,7],sample[0,8])
reference=rc_teacher(sample[0],rbser,rbtser,rsser,
                     CR_K_POINTS_ULT*2-1,CR_K_POINTS_CRC*2-1,
                     CR_REFINE_ITERS,CR_SIMULTANEOUS_REL_TOL)
forecast=rc_predict(sample)
print('Вход:',dict(zip(CR_INPUT_COLUMNS,sample[0])))
print('НДМ: lambda_first, lambda_ult, lambda_crc_if_first, event =',reference[:4])
print('НДМ: sigma_sb_crc / sigma_st_crc, МПа =',reference[4:6])
print('НДМ: sigma_sb_full / sigma_st_full, МПа =',reference[6:8])
print('НДМ: sigma_s_crc (максимальное растяжение), МПа =',
      max(0.,reference[4],reference[5]) if np.all(np.isfinite(reference[4:6])) else np.nan)
print('НДМ: sigma_s_full (максимальное растяжение), МПа =',
      max(0.,reference[6],reference[7]) if np.all(np.isfinite(reference[6:8])) else np.nan)
print('ML : lambda_first=',float(forecast['lambda_first'][0]),
      ', lambda_ult=',float(forecast['lambda_ult'][0]),
      ', lambda_crc_if_first=',float(forecast['lambda_crc_if_first'][0]),
      ', event=',CR_EVENTS[int(forecast['event'][0])])
for key in ('sigma_sb_crc','sigma_st_crc','sigma_s_crc',
            'sigma_sb_full','sigma_st_full','sigma_s_full'):
    print(f'ML: {key}, МПа =',float(forecast[key][0]))
if np.all(np.isfinite(reference[4:6])):
    print('Растянутый ряд при трещинообразовании:',
          'нижний' if reference[4]>=reference[5] else 'верхний')
if np.all(np.isfinite(reference[6:8])):
    print('Растянутый ряд при полной нагрузке:',
          'нижний' if reference[6]>=reference[7] else 'верхний')
else:
    print('sigma_s_full не определена: до lambda=1 трещина не возникла,',
          'достигнут предел прочности или не найдено равновесие.')
print('Вероятности событий:',forecast['probabilities'][0])
print('При λ=1: предел прочности по эталону', 'ВЫПОЛНЕН' if reference[1]>=1 else 'НЕ ВЫПОЛНЕН')
print('При λ=1: до первого предела', 'НЕ ДОШЛИ' if reference[0]>=1 else 'ДОШЛИ')

Вход: {'N_kN_m': np.float64(-168.0), 'My_kNm_m': np.float64(35.0), 'Rb_MPa': np.float64(17.0), 'Eb_MPa': np.float64(32500.0), 'h_mm': np.float64(200.0), 'asb_mm': np.float64(30.0), 'ast_mm': np.float64(30.0), 'Rs_MPa': np.float64(435.0), 'Rsc_MPa': np.float64(400.0), 'Asb_cm2_m': np.float64(8.0), 'Ast_cm2_m': np.float64(8.0)}
НДМ: lambda_first, lambda_ult, lambda_crc_if_first, event = (0.7757709806643565, 2.8083970494253068, 0.7757709806643565, 1)
НДМ: sigma_sb_crc / sigma_st_crc, МПа = (134.17360285791244, -21.873129704882555)
НДМ: sigma_sb_full / sigma_st_full, МПа = (172.95517095574544, -28.195344227253276)
НДМ: sigma_s_crc (максимальное растяжение), МПа = 134.17360285791244
НДМ: sigma_s_full (максимальное растяжение), МПа = 172.95517095574544
ML : lambda_first= 0.770647406578064 , lambda_ult= 2.7982945442199707 , lambda_crc_if_first= 0.770647406578064 , event= CRACK_FIRST
ML: sigma_sb_crc, МПа = 133.28541815280914
ML: sigma_st_crc, МПа = -24.51832965016365
ML: sigma_s_crc, МПа = 13

## 8. Перенос в Rust и статус модели

Архив весов `.npz` содержит `W0..W3`, `b0..b3`, `x_mean`, `x_std`, `y_mean`, `y_std`, `architecture`, `sigma_outputs`, `target_columns`, матрицы бетона/арматуры, названия событий и версию. Матричное умножение: `x @ W + b`; на скрытых слоях **SiLU**, на выходе линейный слой. Первая пара выходов содержит нормированные **логарифмы** индексов, следующие четыре — через `tanh` напряжения `sigma_sb_crc/Rs_ser`, `sigma_st_crc/Rs_ser`, `sigma_sb_full/Rs_ser`, `sigma_st_full/Rs_ser` (значения при полной нагрузке маскируются для нетрещиноватых или разрушившихся состояний), последние три — логиты событий. Для Rust необходимо точно повторить `rc_features` и `rc_predict`; изменения матриц материалов требуют повторной оценки области применимости или переобучения. Веса предыдущей `crack-ray-v1` и `crack-ray-v2-normative` не совместимы с новой постановкой: обучить модель с нуля; `rc_predict` проверяет `model_version`.

Для расчёта трещин, предела прочности и напряжений в арматуре сразу после образования трещины **и** при полной нагрузке непосредственно по НДМ используйте `rc_teacher`. **Обе границы рассчитываются с нормативными материалами**; они различаются учётом растянутого бетона. Его вывод — вычислительный эталон на заданной точности κ-сетки, но не сертифицированное определение экспериментальной последовательности событий. Запрещено использовать прогноз без расчётной проверки как основание для окончательного проектного заключения.
`model_version = crack-ray-v4-normative-crack-and-full-stress`, архитектура `22–188–188–188–9`. Веса версии v3 несовместимы.
